In [ ]:
# ============================================================
# Initially disk rotates simultanously with the particles 
# ============================================================


import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.patches import FancyArrowPatch
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from mpl_toolkits.mplot3d import proj3d
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.integrate import solve_bvp
from IPython.display import Image, Video, display

import matplotlib as mpl

try:
    import imageio_ffmpeg
    mpl.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
except Exception:
    print("[UYARI] imageio_ffmpeg bulunamadı veya ffmpeg yolu atanamadı.")
    print("MP4 kaydı hata verirse: pip install imageio-ffmpeg")


# ============================================================
# 3D Arrow Class
# ============================================================
class Arrow3D(FancyArrowPatch):
    def __init__(self, xs, ys, zs, *args, **kwargs):
        super().__init__((0, 0), (0, 0), *args, **kwargs)
        self._verts3d = xs, ys, zs

    def do_3d_projection(self, renderer=None):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, self.axes.M)
        self.set_positions((xs[0], ys[0]), (xs[1], ys[1]))
        return np.min(zs)


def add_arrow3d(ax, start, end, color="k", lw=2.0, mutation_scale=18, alpha=1.0):
    arrow = Arrow3D(
        [start[0], end[0]],
        [start[1], end[1]],
        [start[2], end[2]],
        arrowstyle="-|>",
        mutation_scale=mutation_scale,
        lw=lw,
        color=color,
        alpha=alpha,
    )
    ax.add_artist(arrow)
    return arrow


# ============================================================
# Transparent rectangular prism
# ============================================================
def add_transparent_box(
    ax,
    xmin, xmax,
    ymin, ymax,
    zmin, zmax,
    face_alpha=0.028,
    edge_alpha=0.22,
):
    vertices = np.array(
        [
            [xmin, ymin, zmin],
            [xmax, ymin, zmin],
            [xmax, ymax, zmin],
            [xmin, ymax, zmin],
            [xmin, ymin, zmax],
            [xmax, ymin, zmax],
            [xmax, ymax, zmax],
            [xmin, ymax, zmax],
        ]
    )

    faces = [
        [vertices[0], vertices[1], vertices[2], vertices[3]],
        [vertices[4], vertices[5], vertices[6], vertices[7]],
        [vertices[0], vertices[1], vertices[5], vertices[4]],
        [vertices[1], vertices[2], vertices[6], vertices[5]],
        [vertices[2], vertices[3], vertices[7], vertices[6]],
        [vertices[3], vertices[0], vertices[4], vertices[7]],
    ]

    box = Poly3DCollection(
        faces,
        facecolors=(0.35, 0.55, 0.85, face_alpha),
        edgecolors=(0.05, 0.15, 0.25, edge_alpha),
        linewidths=0.85,
    )
    ax.add_collection3d(box)

    edges = [
        (0, 1), (1, 2), (2, 3), (3, 0),
        (4, 5), (5, 6), (6, 7), (7, 4),
        (0, 4), (1, 5), (2, 6), (3, 7),
    ]

    for i, j in edges:
        ax.plot(
            [vertices[i, 0], vertices[j, 0]],
            [vertices[i, 1], vertices[j, 1]],
            [vertices[i, 2], vertices[j, 2]],
            color=(0.05, 0.15, 0.25, edge_alpha),
            lw=0.85,
        )


# ============================================================
# Parameters
# ============================================================
R_disk = 5.4
H_disk = 0.65

R_fluid = 5.4
Z_min = 0.0
Z_max = 20.0

box_half = 6.4
box_zmin = 0.0
box_zmax = 20.0

OMEGA = np.pi

DISK_TOP_ALPHA = 0.55

BL_PARTICLE_Z_LIFT = 0.24

BL_R_INNER = 0.40
BL_R_OUTER = R_disk - 0.20

FPS = 8
DURATION = 20
NFRAMES = FPS * DURATION
DT = 1.0 / FPS

OUTPUT_GIF = "rotating_disk_vonkarman_scheduled_inflow.gif"
OUTPUT_MP4 = "rotating_disk_vonkarman_scheduled_inflow.mp4"

ETA_MAX_VK = 12.0
ETA_VIS_SCALE = 1.8


# ============================================================
# von Karman velocity scaling
# ============================================================
RE_VIS = 100.0
DELTA = RE_VIS ** (-0.5)
DELTA2 = DELTA ** 2

# Fiziksel ölçek:
# U = r F(eta)
# V = r G(eta)
# W = delta^2 H(eta)
#
# Silindirik kinematik:
# dr/dt     ~ U = r F(eta)
# dtheta/dt ~ V/r = G(eta)
# dz/dt     ~ W = delta^2 H(eta)
#
# W = delta^2 H görselde çok küçük kalacağı için görsel zaman ölçeğiyle büyütülür.
U_VISUAL_GAIN = 0.045 * OMEGA
THETA_VISUAL_GAIN = OMEGA
W_VISUAL_GAIN = 36.0 * OMEGA

EDGE_FADE_START = 0.88 * BL_R_OUTER

FAR_TRAIL_LEN = 48
VORTEX_TRAIL_LEN = 56
BL_TRAIL_LEN = 34

RED_FADE_SEGMENTS = 90

# Uzak alan parçacıkları için sürekli yeni parçacık enjeksiyonu.
# Aynı parçacık alta indikten sonra yukarı taşınmaz.
FAR_FADE_IN_FRAMES = max(2, int(0.8 * FPS))
FAR_FADE_OUT_HEIGHT = 1.8

FAR_INFLOW_Z_RANGE = (18.6, 19.7)
FAR_INFLOW_INTERVAL_FRAMES = max(1, int(1.15 * FPS))

# Video başında sahnenin boş görünmemesi için başlangıçta var olan parçacık seviyeleri.
FAR_INITIAL_Z_LEVELS = [19.0, 16.8, 14.6, 12.4, 10.2, 8.0]


# ============================================================
# von Karman rotating-disk similarity solution
# ============================================================
def solve_von_karman_rotating_disk():
    eta = np.linspace(0.0, ETA_MAX_VK, 260)

    def ode(eta, y):
        F = y[0]
        Fp = y[1]
        G = y[2]
        Gp = y[3]
        H = y[4]

        Fpp = F * F - G * G + Fp * H
        Gpp = 2.0 * F * G + Gp * H
        Hp = -2.0 * F

        return np.vstack((Fp, Fpp, Gp, Gpp, Hp))

    def bc(ya, yb):
        return np.array(
            [
                ya[0],        # F(0) = 0
                ya[2] - 1.0,  # G(0) = 1
                ya[4],        # H(0) = 0
                yb[0],        # F(infty) = 0
                yb[2],        # G(infty) = 0
            ]
        )

    F_guess = 0.5 * eta * np.exp(-eta)
    Fp_guess = 0.5 * (1.0 - eta) * np.exp(-eta)
    G_guess = np.exp(-eta)
    Gp_guess = -np.exp(-eta)
    H_guess = -0.9 * (1.0 - np.exp(-eta))

    y_guess = np.vstack([F_guess, Fp_guess, G_guess, Gp_guess, H_guess])

    sol = solve_bvp(
        ode,
        bc,
        eta,
        y_guess,
        tol=1e-5,
        max_nodes=20000,
    )

    if not sol.success:
        raise RuntimeError("von Karman BVP çözümü yakınsamadı: " + sol.message)

    return sol


vk_sol = solve_von_karman_rotating_disk()

eta_dense = np.linspace(0.0, ETA_MAX_VK, 800)
vk_dense = vk_sol.sol(eta_dense)
F_dense = vk_dense[0]
H_dense = vk_dense[4]

F_MAX = np.max(np.abs(F_dense))
H_INF_ABS = np.abs(H_dense[-1])


def vk_profiles(z):
    """
    Görsel z koordinatını eta koordinatına map eder.

    Çıktılar:
        Fn : normalize radyal ağırlık [0, 1]
        Gn : azimutal ağırlık [0, 1]
        Hn : normalize eksenel ağırlık [-1, 0]
    """
    z_arr = np.asarray(z, dtype=float)
    eta = np.clip(z_arr / ETA_VIS_SCALE, 0.0, ETA_MAX_VK)

    vals = vk_sol.sol(eta)
    F = vals[0]
    G = vals[2]
    H = vals[4]

    Fn = np.clip(F / F_MAX, 0.0, 1.0)
    Gn = np.clip(G, 0.0, 1.0)
    Hn = np.clip(H / H_INF_ABS, -1.0, 0.0)

    return Fn, Gn, Hn


# ============================================================
# General particle integration with von Karman scaling
# ============================================================
def integrate_particle_trajectory(
    r0,
    th0,
    z0,
    nframes,
    dt,
    z_floor,
    r_min=0.15,
    r_max=R_fluid - 0.12,
    substeps=5,
):
    """
    Genel parçacık entegrasyonu.

    Re-spawn yoktur.
    Sınır tabaka parçacıkları, vortex trail, uzak alan trail ve kırmızı tracer için kullanılır.

        U = r F(eta)
        V = r G(eta)
        W = delta^2 H(eta)
    """
    x = np.empty(nframes)
    y = np.empty(nframes)
    z_hist = np.empty(nframes)
    r_hist = np.empty(nframes)

    r = float(r0)
    th = float(th0)
    z = float(z0)

    x[0] = r * np.cos(th)
    y[0] = r * np.sin(th)
    z_hist[0] = z
    r_hist[0] = r

    dt_sub = dt / substeps

    for i in range(1, nframes):
        for _ in range(substeps):
            Fn, Gn, Hn = vk_profiles(z)

            Fn = float(Fn)
            Gn = float(Gn)
            Hn = float(Hn)

            U = r * Fn
            V_over_r = Gn
            W = DELTA2 * Hn

            dr_dt = U_VISUAL_GAIN * U
            dth_dt = THETA_VISUAL_GAIN * V_over_r
            dz_dt = W_VISUAL_GAIN * W

            r = r + dr_dt * dt_sub
            th = th + dth_dt * dt_sub
            z = z + dz_dt * dt_sub

            r = np.clip(r, r_min, r_max)
            z = np.clip(z, z_floor, Z_max - 0.12)

        x[i] = r * np.cos(th)
        y[i] = r * np.sin(th)
        z_hist[i] = z
        r_hist[i] = r

    return x, y, z_hist, r_hist


# ============================================================
# Far-field particle integration with scheduled one-way injection
# ============================================================
def integrate_scheduled_far_particle_trajectory(
    r0,
    th0,
    z0,
    start_frame,
    nframes,
    dt,
    z_floor,
    r_min=0.15,
    r_max=R_fluid - 0.12,
    substeps=5,
    fade_in_frames=FAR_FADE_IN_FRAMES,
    fade_out_height=FAR_FADE_OUT_HEIGHT,
):
    """
    Uzak alan parçacıkları için tek-yönlü zamanlanmış enjeksiyon.

    Bu fonksiyon aynı parçacığı alta indikten sonra yukarı taşımaz.

    Mantık:
    - frame < start_frame: parçacık görünmez
    - frame >= start_frame: parçacık yukarıdan girer ve aşağı iner
    - z <= z_floor olduğunda parçacık görünmez olur ve video sonuna kadar geri gelmez
    - sahneye sürekli yeni parçacık girişi, farklı start_frame değerleriyle oluşturulan
      farklı artist'ler üzerinden sağlanır

    Hareket:
        U = r F(eta)
        V = r G(eta)
        W = delta^2 H(eta)
    """
    x = np.zeros(nframes)
    y = np.zeros(nframes)
    z_hist = np.zeros(nframes)
    r_hist = np.zeros(nframes)
    vis_hist = np.zeros(nframes)

    r = float(r0)
    th = float(th0)
    z = float(z0)

    dt_sub = dt / substeps
    alive = False

    for i in range(nframes):
        if i < start_frame:
            x[i] = r * np.cos(th)
            y[i] = r * np.sin(th)
            z_hist[i] = z
            r_hist[i] = r
            vis_hist[i] = 0.0
            continue

        if i == start_frame:
            alive = True

        if not alive:
            x[i] = r * np.cos(th)
            y[i] = r * np.sin(th)
            z_hist[i] = z
            r_hist[i] = r
            vis_hist[i] = 0.0
            continue

        # Mevcut konumu kaydet.
        x[i] = r * np.cos(th)
        y[i] = r * np.sin(th)
        z_hist[i] = z
        r_hist[i] = r

        age = i - start_frame

        fade_in = np.clip(age / fade_in_frames, 0.0, 1.0)
        fade_out = np.clip((z - z_floor) / fade_out_height, 0.0, 1.0)

        vis_hist[i] = fade_in * fade_out

        # Bir sonraki frame için konumu ilerlet.
        for _ in range(substeps):
            Fn, Gn, Hn = vk_profiles(z)

            Fn = float(Fn)
            Gn = float(Gn)
            Hn = float(Hn)

            U = r * Fn
            V_over_r = Gn
            W = DELTA2 * Hn

            dr_dt = U_VISUAL_GAIN * U
            dth_dt = THETA_VISUAL_GAIN * V_over_r
            dz_dt = W_VISUAL_GAIN * W

            r = r + dr_dt * dt_sub
            th = th + dth_dt * dt_sub
            z = z + dz_dt * dt_sub

            r = np.clip(r, r_min, r_max)

            if z <= z_floor + 0.04:
                z = z_floor
                alive = False
                break

            z = np.clip(z, z_floor, Z_max - 0.12)

    return x, y, z_hist, r_hist, vis_hist


# ============================================================
# Figure setup
# ============================================================
fig = plt.figure(figsize=(8.5, 6.3), dpi=110)
ax = fig.add_subplot(111, projection="3d")

try:
    ax.computed_zorder = False
except Exception:
    pass

fig.patch.set_facecolor("white")
ax.set_facecolor("white")


# ============================================================
# Static scene
# ============================================================
def draw_static_scene(ax):
    theta = np.linspace(0, 2 * np.pi, 280)
    r = np.linspace(0, R_disk, 115)
    Theta, Rad = np.meshgrid(theta, r)

    X = Rad * np.cos(Theta)
    Y = Rad * np.sin(Theta)

    Z_top = np.zeros_like(X)
    Z_bottom = -H_disk * np.ones_like(X)

    metal = (
        0.17
        + 0.10 * (Rad / R_disk) ** 0.90
        + 0.010 * np.cos(16 * np.pi * Rad / R_disk)
    )
    metal = np.clip(metal, 0.0, 1.0)

    face_colors_top = cm.Greys(metal)
    face_colors_top[..., 3] = DISK_TOP_ALPHA

    ax.plot_surface(
        X, Y, Z_top,
        facecolors=face_colors_top,
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=DISK_TOP_ALPHA,
        zorder=1,
    )

    ax.plot_surface(
        X, Y, Z_bottom,
        color="#5f5f5f",
        linewidth=0,
        antialiased=True,
        shade=True,
        alpha=1.0,
        zorder=1,
    )

    Theta_side, Z_side = np.meshgrid(theta, np.linspace(-H_disk, 0.0, 36))
    X_side = R_disk * np.cos(Theta_side)
    Y_side = R_disk * np.sin(Theta_side)

    ax.plot_surface(
        X_side, Y_side, Z_side,
        color="#4f4f4f",
        linewidth=0,
        antialiased=True,
        shade=True,
        alpha=1.0,
        zorder=1,
    )

    ax.plot(
        R_disk * np.cos(theta),
        R_disk * np.sin(theta),
        np.zeros_like(theta),
        color="#2f2f2f",
        lw=1.7,
        zorder=5,
    )

    ax.plot(
        R_disk * np.cos(theta),
        R_disk * np.sin(theta),
        -H_disk * np.ones_like(theta),
        color="#444444",
        lw=1.1,
        zorder=5,
    )

    for rr in np.linspace(0.8, R_disk - 0.35, 8):
        ax.plot(
            rr * np.cos(theta),
            rr * np.sin(theta),
            0.025 * np.ones_like(theta),
            color="white",
            lw=0.50,
            alpha=0.10,
            zorder=3,
        )

    # Su silindiri
    theta_c = np.linspace(0, 2 * np.pi, 190)
    z_c = np.linspace(Z_min, Z_max, 65)

    Theta_c, Z_c = np.meshgrid(theta_c, z_c)
    X_c = R_fluid * np.cos(Theta_c)
    Y_c = R_fluid * np.sin(Theta_c)

    ax.plot_surface(
        X_c, Y_c, Z_c,
        color="#6ec6ff",
        alpha=0.075,
        linewidth=0,
        antialiased=True,
        shade=False,
        zorder=0,
    )

    r_top = np.linspace(0, R_fluid, 60)
    theta_top = np.linspace(0, 2 * np.pi, 160)
    Theta_top, Rad_top = np.meshgrid(theta_top, r_top)

    X_top = Rad_top * np.cos(Theta_top)
    Y_top = Rad_top * np.sin(Theta_top)
    Z_top_water = Z_max * np.ones_like(X_top)

    ax.plot_surface(
        X_top, Y_top, Z_top_water,
        color="#aee6ff",
        alpha=0.075,
        linewidth=0,
        antialiased=True,
        shade=False,
        zorder=0,
    )

    ax.plot(
        R_fluid * np.cos(theta_c),
        R_fluid * np.sin(theta_c),
        Z_min * np.ones_like(theta_c),
        color="#2d9cdb",
        lw=1.45,
        alpha=0.78,
        zorder=4,
    )

    ax.plot(
        R_fluid * np.cos(theta_c),
        R_fluid * np.sin(theta_c),
        Z_max * np.ones_like(theta_c),
        color="#2d9cdb",
        lw=1.35,
        alpha=0.65,
        zorder=4,
    )

    for th in np.linspace(0, 2 * np.pi, 10, endpoint=False):
        ax.plot(
            [R_fluid * np.cos(th), R_fluid * np.cos(th)],
            [R_fluid * np.sin(th), R_fluid * np.sin(th)],
            [Z_min, Z_max],
            color="#56b4ef",
            lw=0.55,
            alpha=0.16,
            zorder=0,
        )

    add_transparent_box(
        ax,
        xmin=-box_half,
        xmax=box_half,
        ymin=-box_half,
        ymax=box_half,
        zmin=box_zmin,
        zmax=box_zmax,
        face_alpha=0.026,
        edge_alpha=0.22,
    )

    origin = np.array([0.0, 0.0, 0.0])

    add_arrow3d(
        ax,
        start=origin,
        end=(0.0, 0.0, 22.0),
        color="black",
        lw=2.3,
        mutation_scale=18,
    )
    ax.text(0.25, 0.20, 22.7, r"$z$", fontsize=15, color="black")

    add_arrow3d(
        ax,
        start=origin,
        end=(7.3, 0.0, 0.0),
        color="black",
        lw=2.3,
        mutation_scale=18,
    )
    ax.text(7.75, 0.0, 0.25, r"$r$", fontsize=15, color="black")

    # Statik dönme oku
    rot_R = 1.15
    rot_z = 18.5

    phi = np.linspace(np.deg2rad(35), np.deg2rad(330), 150)
    x_rot = rot_R * np.cos(phi)
    y_rot = rot_R * np.sin(phi)
    z_rot = rot_z * np.ones_like(phi)

    ax.plot(x_rot, y_rot, z_rot, color="black", lw=1.9)

    add_arrow3d(
        ax,
        start=(x_rot[-8], y_rot[-8], z_rot[-8]),
        end=(x_rot[-1], y_rot[-1], z_rot[-1]),
        color="black",
        lw=1.9,
        mutation_scale=16,
    )

    ax.set_xlim(-7.4, 8.2)
    ax.set_ylim(-7.2, 7.2)
    ax.set_zlim(-1.2, 23.5)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

    ax.grid(False)

    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis.pane.fill = False
        axis.pane.set_edgecolor((1, 1, 1, 0))
        axis.line.set_color((1, 1, 1, 0))

    ax.set_box_aspect((1.35, 1.25, 1.55))
    ax.view_init(elev=23, azim=-50)


draw_static_scene(ax)


# ============================================================
# Dynamic artists
# ============================================================

# ------------------------------------------------------------
# 1) Disk motion markers
# ------------------------------------------------------------
disk_marker_lines = []
disk_marker_offsets = np.linspace(0, 2 * np.pi, 5, endpoint=False)

for _ in range(5):
    line, = ax.plot(
        [], [], [],
        color="#111111",
        lw=1.55,
        alpha=0.62,
        solid_capstyle="round",
        zorder=900,
    )
    disk_marker_lines.append(line)

disk_rim_marker_lines = []
rim_marker_offsets = np.linspace(0, 2 * np.pi, 10, endpoint=False)

for _ in range(10):
    line, = ax.plot(
        [], [], [],
        color="#101010",
        lw=1.35,
        alpha=0.72,
        solid_capstyle="round",
        zorder=902,
    )
    disk_rim_marker_lines.append(line)

disk_marker_dot, = ax.plot(
    [], [], [],
    linestyle="None",
    marker="o",
    markersize=3.3,
    color="#101010",
    alpha=0.90,
    zorder=903,
)


# ------------------------------------------------------------
# 2) Uzak alan parçacıkları: zamanlanmış sürekli enjeksiyon
# ------------------------------------------------------------
rng = np.random.default_rng(42)

far_particle_specs = []
far_particle_artists = []
far_particle_trajs = []

far_columns = [
    (1.15, np.deg2rad(18)),
    (1.75, np.deg2rad(72)),
    (2.35, np.deg2rad(128)),
    (2.95, np.deg2rad(188)),
    (3.55, np.deg2rad(242)),
    (4.15, np.deg2rad(304)),
    (4.75, np.deg2rad(36)),
    (5.05, np.deg2rad(160)),
]

# 2A) Video başında zaten alanda bulunan parçacıklar
for ic, (r_col, th_col) in enumerate(far_columns):
    for iz, z0 in enumerate(FAR_INITIAL_Z_LEVELS):
        r0 = r_col * (0.985 + 0.030 * rng.random())
        th0 = th_col + 0.025 * rng.standard_normal()
        start_frame = 0

        far_particle_specs.append((r0, th0, z0, start_frame))

# 2B) Video boyunca yukarıdan yeni giren parçacıklar
spawn_frames = np.arange(
    FAR_INFLOW_INTERVAL_FRAMES,
    NFRAMES,
    FAR_INFLOW_INTERVAL_FRAMES,
    dtype=int
)

for start_frame in spawn_frames:
    for ic, (r_col, th_col) in enumerate(far_columns):
        # Her enjeksiyonda tüm kolonlardan parçacık vermek fazla kalabalık olabilir.
        # Kolonlar dönüşümlü aktif.
        if (ic + start_frame // FAR_INFLOW_INTERVAL_FRAMES) % 2 != 0:
            continue

        r0 = r_col * (0.985 + 0.030 * rng.random())
        th0 = th_col + 0.025 * rng.standard_normal()
        z0 = rng.uniform(FAR_INFLOW_Z_RANGE[0], FAR_INFLOW_Z_RANGE[1])

        far_particle_specs.append((r0, th0, z0, start_frame))

# Artist ve trajeler
for r0, th0, z0, start_frame in far_particle_specs:
    artist, = ax.plot(
        [], [], [],
        linestyle="None",
        marker="o",
        markersize=3.1,
        color="#0d6efd",
        alpha=0.0,
        zorder=650,
    )
    far_particle_artists.append(artist)

    traj = integrate_scheduled_far_particle_trajectory(
        r0=r0,
        th0=th0,
        z0=z0,
        start_frame=start_frame,
        nframes=NFRAMES,
        dt=DT,
        z_floor=0.38,
        r_min=0.18,
        r_max=R_fluid - 0.12,
        substeps=5,
        fade_in_frames=FAR_FADE_IN_FRAMES,
        fade_out_height=FAR_FADE_OUT_HEIGHT,
    )

    far_particle_trajs.append(traj)


# ------------------------------------------------------------
# 3) Sınır tabaka / disk üstü parçacıkları
# ------------------------------------------------------------
bl_particle_specs = []
bl_particle_artists = []
bl_particle_trajs = []

bl_z_levels = np.array([0.08, 0.15, 0.25, 0.40, 0.60, 0.85, 1.15])
bl_radii = np.array([0.75, 1.15, 1.60, 2.10, 2.70, 3.30, 3.95, 4.55, 5.00])

for iz, z0 in enumerate(bl_z_levels):
    for ir, r0 in enumerate(bl_radii):
        if r0 > 4.3:
            n_theta = 2
        else:
            n_theta = 2 if r0 < 2.2 else 3

        for k in range(n_theta):
            th0 = (
                2.0 * np.pi * k / n_theta
                + 0.54 * ir
                + 0.28 * iz
                + 0.10 * rng.random()
            )

            bl_particle_specs.append((r0, th0, z0))

            _, G0, _ = vk_profiles(z0)
            marker_size = 4.1 + 1.9 * float(G0)

            particle, = ax.plot(
                [], [], [],
                linestyle="None",
                marker="o",
                markersize=marker_size,
                markerfacecolor="#00b7ff",
                markeredgecolor="#005f88",
                markeredgewidth=0.35,
                alpha=1.0,
                zorder=1500,
            )
            bl_particle_artists.append(particle)

for r0, th0, z0 in bl_particle_specs:
    traj = integrate_particle_trajectory(
        r0=r0,
        th0=th0,
        z0=BL_PARTICLE_Z_LIFT + z0,
        nframes=NFRAMES,
        dt=DT,
        z_floor=BL_PARTICLE_Z_LIFT + 0.05,
        r_min=BL_R_INNER,
        r_max=BL_R_OUTER,
        substeps=5,
    )
    bl_particle_trajs.append(traj)


# ------------------------------------------------------------
# 4A) Uzak alan aşağı yönlü trail çizgileri
# ------------------------------------------------------------
far_trail_seed_specs = [
    (2.10, np.deg2rad(35),  18.6),
    (2.70, np.deg2rad(90),  16.8),
    (3.30, np.deg2rad(145), 15.0),
    (3.85, np.deg2rad(205), 13.2),
    (4.35, np.deg2rad(265), 11.4),
    (4.85, np.deg2rad(325), 9.6),
]

far_trail_lines = []
far_trail_trajs = []

for _ in far_trail_seed_specs:
    line, = ax.plot(
        [], [], [],
        color="#3aa8f5",
        lw=1.15,
        alpha=0.56,
        zorder=470,
    )
    far_trail_lines.append(line)

for r0, th0, z0 in far_trail_seed_specs:
    traj = integrate_particle_trajectory(
        r0=r0,
        th0=th0,
        z0=z0,
        nframes=NFRAMES,
        dt=DT,
        z_floor=0.38,
        r_min=0.18,
        r_max=R_fluid - 0.12,
        substeps=5,
    )
    far_trail_trajs.append(traj)


# ------------------------------------------------------------
# 4A-2) Vortex / spiral pathline çizgileri
# ------------------------------------------------------------
vortex_trail_seed_specs = [
    (1.35, np.deg2rad(20),  4.20),
    (2.05, np.deg2rad(85),  3.70),
    (2.85, np.deg2rad(150), 3.15),
    (3.65, np.deg2rad(215), 2.55),
    (4.35, np.deg2rad(285), 2.05),
]

vortex_trail_lines = []
vortex_trail_trajs = []

for _ in vortex_trail_seed_specs:
    line, = ax.plot(
        [], [], [],
        color="#57d7ff",
        lw=1.35,
        alpha=0.58,
        zorder=900,
    )
    vortex_trail_lines.append(line)

for r0, th0, z0 in vortex_trail_seed_specs:
    traj = integrate_particle_trajectory(
        r0=r0,
        th0=th0,
        z0=z0,
        nframes=NFRAMES,
        dt=DT,
        z_floor=0.38,
        r_min=0.18,
        r_max=R_fluid - 0.12,
        substeps=6,
    )
    vortex_trail_trajs.append(traj)


# ------------------------------------------------------------
# 4B) Azaltılmış sınır tabaka trail çizgileri
# ------------------------------------------------------------
bl_trail_seed_specs = [
    (1.05, np.deg2rad(25),  BL_PARTICLE_Z_LIFT + 1.05),
    (1.55, np.deg2rad(95),  BL_PARTICLE_Z_LIFT + 0.82),
    (2.05, np.deg2rad(165), BL_PARTICLE_Z_LIFT + 0.60),
    (3.15, np.deg2rad(240), BL_PARTICLE_Z_LIFT + 0.42),
    (3.85, np.deg2rad(315), BL_PARTICLE_Z_LIFT + 0.26),
]

bl_trail_lines = []
bl_trail_trajs = []

for _ in bl_trail_seed_specs:
    line, = ax.plot(
        [], [], [],
        color="#00aeea",
        lw=1.15,
        alpha=0.64,
        zorder=1300,
    )
    bl_trail_lines.append(line)

for r0, th0, z0 in bl_trail_seed_specs:
    traj = integrate_particle_trajectory(
        r0=r0,
        th0=th0,
        z0=z0,
        nframes=NFRAMES,
        dt=DT,
        z_floor=BL_PARTICLE_Z_LIFT + 0.05,
        r_min=BL_R_INNER,
        r_max=BL_R_OUTER,
        substeps=5,
    )
    bl_trail_trajs.append(traj)


# ------------------------------------------------------------
# 5) Kırmızı tracer parçacık + fading trail segmentleri
# ------------------------------------------------------------
RED_PARTICLE_R0 = 2.25
RED_PARTICLE_THETA0 = np.deg2rad(58.0)
RED_PARTICLE_Z0 = 15.0
RED_PARTICLE_Z_MIN = BL_PARTICLE_Z_LIFT + 0.05

red_traj = integrate_particle_trajectory(
    r0=RED_PARTICLE_R0,
    th0=RED_PARTICLE_THETA0,
    z0=RED_PARTICLE_Z0,
    nframes=NFRAMES,
    dt=DT,
    z_floor=RED_PARTICLE_Z_MIN,
    r_min=0.18,
    r_max=R_fluid - 0.12,
    substeps=6,
)

red_fade_lines = []

for i in range(RED_FADE_SEGMENTS):
    line, = ax.plot(
        [], [], [],
        color="red",
        lw=1.0,
        alpha=0.0,
        zorder=2500 + i,
    )
    red_fade_lines.append(line)

red_particle_artist, = ax.plot(
    [], [], [],
    linestyle="None",
    marker="o",
    markersize=7.5,
    markerfacecolor="red",
    markeredgecolor="#600000",
    markeredgewidth=0.8,
    alpha=1.0,
    zorder=2700,
)


# ============================================================
# Animation functions
# ============================================================
def init():
    for line in disk_marker_lines:
        line.set_data([], [])
        line.set_3d_properties([])

    for line in disk_rim_marker_lines:
        line.set_data([], [])
        line.set_3d_properties([])

    disk_marker_dot.set_data([], [])
    disk_marker_dot.set_3d_properties([])

    for artist in far_particle_artists:
        artist.set_data([], [])
        artist.set_3d_properties([])
        artist.set_alpha(0.0)

    for artist in bl_particle_artists:
        artist.set_data([], [])
        artist.set_3d_properties([])

    for line in far_trail_lines:
        line.set_data([], [])
        line.set_3d_properties([])

    for line in vortex_trail_lines:
        line.set_data([], [])
        line.set_3d_properties([])

    for line in bl_trail_lines:
        line.set_data([], [])
        line.set_3d_properties([])

    for line in red_fade_lines:
        line.set_data([], [])
        line.set_3d_properties([])
        line.set_alpha(0.0)

    red_particle_artist.set_data([], [])
    red_particle_artist.set_3d_properties([])

    return (
        disk_marker_lines
        + disk_rim_marker_lines
        + [disk_marker_dot]
        + far_particle_artists
        + bl_particle_artists
        + far_trail_lines
        + vortex_trail_lines
        + bl_trail_lines
        + red_fade_lines
        + [red_particle_artist]
    )


def update(frame):
    ax.view_init(elev=23.0, azim=-50.0)

    t = frame / FPS

    # --------------------------------------------------------
    # Disk marker'ları
    # --------------------------------------------------------
    rr = np.linspace(0.95, 4.85, 54)
    z_marker = 0.068

    for i, (line, offset) in enumerate(zip(disk_marker_lines, disk_marker_offsets)):
        th = OMEGA * t + offset

        rr_mod = rr + 0.035 * np.sin(2.2 * rr + 0.8 * i)

        x = rr_mod * np.cos(th)
        y = rr_mod * np.sin(th)
        z = np.full_like(rr_mod, z_marker)

        line.set_data(x, y)
        line.set_3d_properties(z)

    rim_span = np.deg2rad(8.5)
    rim_theta_local = np.linspace(-rim_span, rim_span, 12)
    rim_radius = 5.16
    rim_z = 0.088

    for line, offset in zip(disk_rim_marker_lines, rim_marker_offsets):
        th_center = OMEGA * t + offset
        ths = th_center + rim_theta_local

        x = rim_radius * np.cos(ths)
        y = rim_radius * np.sin(ths)
        z = np.full_like(ths, rim_z)

        line.set_data(x, y)
        line.set_3d_properties(z)

    dot_r = 4.95
    dot_th = OMEGA * t + np.pi / 8

    x_d = dot_r * np.cos(dot_th)
    y_d = dot_r * np.sin(dot_th)
    z_d = 0.090

    disk_marker_dot.set_data([x_d], [y_d])
    disk_marker_dot.set_3d_properties([z_d])

    # --------------------------------------------------------
    # Uzak alan parçacıkları
    # --------------------------------------------------------
    for artist, traj in zip(far_particle_artists, far_particle_trajs):
        xh, yh, zh, rh, vh = traj

        artist.set_data([xh[frame]], [yh[frame]])
        artist.set_3d_properties([zh[frame]])

        _, G_now, _ = vk_profiles(zh[frame])
        base_alpha = np.clip(0.42 + 0.46 * float(G_now), 0.34, 0.88)

        artist.set_alpha(base_alpha * vh[frame])

    # --------------------------------------------------------
    # Sınır tabaka parçacıkları
    # --------------------------------------------------------
    for artist, traj in zip(bl_particle_artists, bl_particle_trajs):
        xh, yh, zh, rh = traj

        artist.set_data([xh[frame]], [yh[frame]])
        artist.set_3d_properties([zh[frame]])

        r_now = rh[frame]

        if r_now <= EDGE_FADE_START:
            alpha_now = 1.0
        else:
            alpha_now = 1.0 - (r_now - EDGE_FADE_START) / (BL_R_OUTER - EDGE_FADE_START)
            alpha_now = np.clip(alpha_now, 0.15, 1.0)

        artist.set_alpha(alpha_now)

    # --------------------------------------------------------
    # Uzak alan trail'leri
    # --------------------------------------------------------
    for line, traj in zip(far_trail_lines, far_trail_trajs):
        xh, yh, zh, rh = traj
        i0 = max(0, frame - FAR_TRAIL_LEN)

        line.set_data(xh[i0:frame + 1], yh[i0:frame + 1])
        line.set_3d_properties(zh[i0:frame + 1])

    # --------------------------------------------------------
    # Vortex / spiral pathline çizgileri
    # --------------------------------------------------------
    for line, traj in zip(vortex_trail_lines, vortex_trail_trajs):
        xh, yh, zh, rh = traj
        i0 = max(0, frame - VORTEX_TRAIL_LEN)

        line.set_data(xh[i0:frame + 1], yh[i0:frame + 1])
        line.set_3d_properties(zh[i0:frame + 1])

    # --------------------------------------------------------
    # Sınır tabaka trail'leri
    # --------------------------------------------------------
    for line, traj in zip(bl_trail_lines, bl_trail_trajs):
        xh, yh, zh, rh = traj
        i0 = max(0, frame - BL_TRAIL_LEN)

        line.set_data(xh[i0:frame + 1], yh[i0:frame + 1])
        line.set_3d_properties(zh[i0:frame + 1])

    # --------------------------------------------------------
    # Kırmızı tracer fading trail
    # --------------------------------------------------------
    xr, yr, zr, rr = red_traj

    for line in red_fade_lines:
        line.set_data([], [])
        line.set_3d_properties([])
        line.set_alpha(0.0)

    n_available = max(0, frame)
    n_segments = min(RED_FADE_SEGMENTS, n_available)

    start_idx = max(0, frame - n_segments)

    for j in range(n_segments):
        idx0 = start_idx + j
        idx1 = idx0 + 1

        if idx1 > frame:
            break

        line = red_fade_lines[j]

        line.set_data(xr[idx0:idx1 + 1], yr[idx0:idx1 + 1])
        line.set_3d_properties(zr[idx0:idx1 + 1])

        q = (j + 1) / max(1, n_segments)

        line.set_alpha(0.10 + 0.82 * q)
        line.set_linewidth(0.85 + 1.85 * q)

    red_particle_artist.set_data([xr[frame]], [yr[frame]])
    red_particle_artist.set_3d_properties([zr[frame]])

    return (
        disk_marker_lines
        + disk_rim_marker_lines
        + [disk_marker_dot]
        + far_particle_artists
        + bl_particle_artists
        + far_trail_lines
        + vortex_trail_lines
        + bl_trail_lines
        + red_fade_lines
        + [red_particle_artist]
    )


ani = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    init_func=init,
    interval=1000 / FPS,
    blit=False,
    repeat=True,
)


# ============================================================
# Save and display GIF + MP4
# ============================================================
ani.save(
    OUTPUT_GIF,
    writer=PillowWriter(fps=FPS),
    dpi=85,
)

try:
    mp4_writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=3000,
        extra_args=["-pix_fmt", "yuv420p"]
    )

    ani.save(
        OUTPUT_MP4,
        writer=mp4_writer,
        dpi=120,
    )

    mp4_ok = True

except Exception as e:
    mp4_ok = False
    print("[UYARI] MP4 kaydedilemedi.")
    print("Hata:", e)
    print("GIF yine kaydedildi. MP4 için imageio-ffmpeg/ffmpeg ayarlarını kontrol et.")

plt.close(fig)

print(f"GIF kaydedildi: {OUTPUT_GIF}")

if mp4_ok:
    print(f"MP4 kaydedildi: {OUTPUT_MP4}")

display(Image(filename=OUTPUT_GIF))

if mp4_ok:
    display(Video(OUTPUT_MP4, embed=True))

In [ ]:
# ------------------------------------------------
# The disk initally 3 sec wait for rotation and then it starts rotate with smoothly
# ------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.patches import FancyArrowPatch
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from mpl_toolkits.mplot3d import proj3d
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.integrate import solve_bvp
from IPython.display import Image, Video, display

import matplotlib as mpl

try:
    import imageio_ffmpeg
    mpl.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
except Exception:
    print("[UYARI] imageio_ffmpeg bulunamadı veya ffmpeg yolu atanamadı.")
    print("MP4 kaydı hata verirse: pip install imageio-ffmpeg")


# ============================================================
# 3D Arrow Class
# ============================================================
class Arrow3D(FancyArrowPatch):
    def __init__(self, xs, ys, zs, *args, **kwargs):
        super().__init__((0, 0), (0, 0), *args, **kwargs)
        self._verts3d = xs, ys, zs

    def do_3d_projection(self, renderer=None):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, self.axes.M)
        self.set_positions((xs[0], ys[0]), (xs[1], ys[1]))
        return np.min(zs)


def add_arrow3d(ax, start, end, color="k", lw=2.0, mutation_scale=18, alpha=1.0):
    arrow = Arrow3D(
        [start[0], end[0]],
        [start[1], end[1]],
        [start[2], end[2]],
        arrowstyle="-|>",
        mutation_scale=mutation_scale,
        lw=lw,
        color=color,
        alpha=alpha,
    )
    ax.add_artist(arrow)
    return arrow


# ============================================================
# Transparent rectangular prism
# ============================================================
def add_transparent_box(
    ax,
    xmin, xmax,
    ymin, ymax,
    zmin, zmax,
    face_alpha=0.028,
    edge_alpha=0.22,
):
    vertices = np.array(
        [
            [xmin, ymin, zmin],
            [xmax, ymin, zmin],
            [xmax, ymax, zmin],
            [xmin, ymax, zmin],
            [xmin, ymin, zmax],
            [xmax, ymin, zmax],
            [xmax, ymax, zmax],
            [xmin, ymax, zmax],
        ]
    )

    faces = [
        [vertices[0], vertices[1], vertices[2], vertices[3]],
        [vertices[4], vertices[5], vertices[6], vertices[7]],
        [vertices[0], vertices[1], vertices[5], vertices[4]],
        [vertices[1], vertices[2], vertices[6], vertices[5]],
        [vertices[2], vertices[3], vertices[7], vertices[6]],
        [vertices[3], vertices[0], vertices[4], vertices[7]],
    ]

    box = Poly3DCollection(
        faces,
        facecolors=(0.35, 0.55, 0.85, face_alpha),
        edgecolors=(0.05, 0.15, 0.25, edge_alpha),
        linewidths=0.85,
    )
    ax.add_collection3d(box)

    edges = [
        (0, 1), (1, 2), (2, 3), (3, 0),
        (4, 5), (5, 6), (6, 7), (7, 4),
        (0, 4), (1, 5), (2, 6), (3, 7),
    ]

    for i, j in edges:
        ax.plot(
            [vertices[i, 0], vertices[j, 0]],
            [vertices[i, 1], vertices[j, 1]],
            [vertices[i, 2], vertices[j, 2]],
            color=(0.05, 0.15, 0.25, edge_alpha),
            lw=0.85,
        )


# ============================================================
# Parameters
# ============================================================
R_disk = 5.4
H_disk = 0.65

R_fluid = 5.4
Z_min = 0.0
Z_max = 20.0

box_half = 6.4
box_zmin = 0.0
box_zmax = 20.0

OMEGA = np.pi

DISK_TOP_ALPHA = 0.55

BL_PARTICLE_Z_LIFT = 0.24

BL_R_INNER = 0.40
BL_R_OUTER = R_disk - 0.20

FPS = 8
DURATION = 8
NFRAMES = FPS * DURATION
DT = 1.0 / FPS

OUTPUT_GIF = "rotating_disk_vonkarman_startup_spinup.gif"
OUTPUT_MP4 = "rotating_disk_vonkarman_startup_spinup.mp4"

ETA_MAX_VK = 12.0
ETA_VIS_SCALE = 1.8


# ============================================================
# Disk start-up / spin-up timing
# ============================================================
# İlk 3 saniye disk durur. Sonra 4 saniyede yumuşak biçimde tam hıza çıkar.
STARTUP_HOLD_SECONDS = 3.0
SPINUP_SECONDS = 4.0

STARTUP_HOLD_FRAMES = int(STARTUP_HOLD_SECONDS * FPS)
SPINUP_FRAMES = int(SPINUP_SECONDS * FPS)


def smoothstep(u):
    """
    0 -> 1 aralığında yumuşak hızlanma profili.
    Başta ve sonda türev sıfıra yakın olduğu için ani hız sıçraması vermez.
    """
    u = np.clip(u, 0.0, 1.0)
    return 3.0 * u**2 - 2.0 * u**3


def spin_factor(frame):
    """
    Anlık normalize disk hızı:
        0 : disk duruyor
        1 : disk tam hızda
    """
    if frame < STARTUP_HOLD_FRAMES:
        return 0.0

    ramp_frame = frame - STARTUP_HOLD_FRAMES

    if ramp_frame < SPINUP_FRAMES:
        return smoothstep(ramp_frame / SPINUP_FRAMES)

    return 1.0


# Her gerçek video frame'i için efektif akış zamanı.
# İlk 3 saniyede sabit kalır; sonra yavaşça hızlanır.
SPIN_FACTORS = np.array([spin_factor(k) for k in range(NFRAMES)])

FLOW_CLOCK_FRAMES = np.zeros(NFRAMES)
for k in range(1, NFRAMES):
    FLOW_CLOCK_FRAMES[k] = FLOW_CLOCK_FRAMES[k - 1] + SPIN_FACTORS[k - 1]

FLOW_CLOCK_FRAMES = np.clip(FLOW_CLOCK_FRAMES, 0.0, NFRAMES - 1.0)
FLOW_CLOCK_TIME = FLOW_CLOCK_FRAMES / FPS


# ============================================================
# von Karman velocity scaling
# ============================================================
RE_VIS = 100.0
DELTA = RE_VIS ** (-0.5)
DELTA2 = DELTA ** 2

# Fiziksel ölçek:
# U = r F(eta)
# V = r G(eta)
# W = delta^2 H(eta)
#
# Silindirik kinematik:
# dr/dt     ~ U = r F(eta)
# dtheta/dt ~ V/r = G(eta)
# dz/dt     ~ W = delta^2 H(eta)
#
# W = delta^2 H görselde çok küçük kalacağı için görsel zaman ölçeğiyle büyütülür.
U_VISUAL_GAIN = 0.045 * OMEGA
THETA_VISUAL_GAIN = OMEGA
W_VISUAL_GAIN = 36.0 * OMEGA

EDGE_FADE_START = 0.88 * BL_R_OUTER

FAR_TRAIL_LEN = 48
VORTEX_TRAIL_LEN = 56
BL_TRAIL_LEN = 34

RED_FADE_SEGMENTS = 90

# Uzak alan parçacıkları için sürekli yeni parçacık enjeksiyonu.
# Aynı parçacık alta indikten sonra yukarı taşınmaz.
FAR_FADE_IN_FRAMES = max(2, int(0.8 * FPS))
FAR_FADE_OUT_HEIGHT = 1.8

FAR_INFLOW_Z_RANGE = (18.6, 19.7)
FAR_INFLOW_INTERVAL_FRAMES = max(1, int(1.15 * FPS))

# Video başında sahnenin boş görünmemesi için başlangıçta var olan parçacık seviyeleri.
FAR_INITIAL_Z_LEVELS = [19.0, 16.8, 14.6, 12.4, 10.2, 8.0]


# ============================================================
# von Karman rotating-disk similarity solution
# ============================================================
def solve_von_karman_rotating_disk():
    eta = np.linspace(0.0, ETA_MAX_VK, 260)

    def ode(eta, y):
        F = y[0]
        Fp = y[1]
        G = y[2]
        Gp = y[3]
        H = y[4]

        Fpp = F * F - G * G + Fp * H
        Gpp = 2.0 * F * G + Gp * H
        Hp = -2.0 * F

        return np.vstack((Fp, Fpp, Gp, Gpp, Hp))

    def bc(ya, yb):
        return np.array(
            [
                ya[0],        # F(0) = 0
                ya[2] - 1.0,  # G(0) = 1
                ya[4],        # H(0) = 0
                yb[0],        # F(infty) = 0
                yb[2],        # G(infty) = 0
            ]
        )

    F_guess = 0.5 * eta * np.exp(-eta)
    Fp_guess = 0.5 * (1.0 - eta) * np.exp(-eta)
    G_guess = np.exp(-eta)
    Gp_guess = -np.exp(-eta)
    H_guess = -0.9 * (1.0 - np.exp(-eta))

    y_guess = np.vstack([F_guess, Fp_guess, G_guess, Gp_guess, H_guess])

    sol = solve_bvp(
        ode,
        bc,
        eta,
        y_guess,
        tol=1e-5,
        max_nodes=20000,
    )

    if not sol.success:
        raise RuntimeError("von Karman BVP çözümü yakınsamadı: " + sol.message)

    return sol


vk_sol = solve_von_karman_rotating_disk()

eta_dense = np.linspace(0.0, ETA_MAX_VK, 800)
vk_dense = vk_sol.sol(eta_dense)
F_dense = vk_dense[0]
H_dense = vk_dense[4]

F_MAX = np.max(np.abs(F_dense))
H_INF_ABS = np.abs(H_dense[-1])


def vk_profiles(z):
    """
    Görsel z koordinatını eta koordinatına map eder.

    Çıktılar:
        Fn : normalize radyal ağırlık [0, 1]
        Gn : azimutal ağırlık [0, 1]
        Hn : normalize eksenel ağırlık [-1, 0]
    """
    z_arr = np.asarray(z, dtype=float)
    eta = np.clip(z_arr / ETA_VIS_SCALE, 0.0, ETA_MAX_VK)

    vals = vk_sol.sol(eta)
    F = vals[0]
    G = vals[2]
    H = vals[4]

    Fn = np.clip(F / F_MAX, 0.0, 1.0)
    Gn = np.clip(G, 0.0, 1.0)
    Hn = np.clip(H / H_INF_ABS, -1.0, 0.0)

    return Fn, Gn, Hn


# ============================================================
# General particle integration with von Karman scaling
# ============================================================
def integrate_particle_trajectory(
    r0,
    th0,
    z0,
    nframes,
    dt,
    z_floor,
    r_min=0.15,
    r_max=R_fluid - 0.12,
    substeps=5,
):
    """
    Genel parçacık entegrasyonu.

    Re-spawn yoktur.
    Sınır tabaka parçacıkları, vortex trail, uzak alan trail ve kırmızı tracer için kullanılır.

        U = r F(eta)
        V = r G(eta)
        W = delta^2 H(eta)
    """
    x = np.empty(nframes)
    y = np.empty(nframes)
    z_hist = np.empty(nframes)
    r_hist = np.empty(nframes)

    r = float(r0)
    th = float(th0)
    z = float(z0)

    x[0] = r * np.cos(th)
    y[0] = r * np.sin(th)
    z_hist[0] = z
    r_hist[0] = r

    dt_sub = dt / substeps

    for i in range(1, nframes):
        for _ in range(substeps):
            Fn, Gn, Hn = vk_profiles(z)

            Fn = float(Fn)
            Gn = float(Gn)
            Hn = float(Hn)

            U = r * Fn
            V_over_r = Gn
            W = DELTA2 * Hn

            dr_dt = U_VISUAL_GAIN * U
            dth_dt = THETA_VISUAL_GAIN * V_over_r
            dz_dt = W_VISUAL_GAIN * W

            r = r + dr_dt * dt_sub
            th = th + dth_dt * dt_sub
            z = z + dz_dt * dt_sub

            r = np.clip(r, r_min, r_max)
            z = np.clip(z, z_floor, Z_max - 0.12)

        x[i] = r * np.cos(th)
        y[i] = r * np.sin(th)
        z_hist[i] = z
        r_hist[i] = r

    return x, y, z_hist, r_hist


# ============================================================
# Far-field particle integration with scheduled one-way injection
# ============================================================
def integrate_scheduled_far_particle_trajectory(
    r0,
    th0,
    z0,
    start_frame,
    nframes,
    dt,
    z_floor,
    r_min=0.15,
    r_max=R_fluid - 0.12,
    substeps=5,
    fade_in_frames=FAR_FADE_IN_FRAMES,
    fade_out_height=FAR_FADE_OUT_HEIGHT,
):
    """
    Uzak alan parçacıkları için tek-yönlü zamanlanmış enjeksiyon.

    Bu fonksiyon aynı parçacığı alta indikten sonra yukarı taşımaz.

    Mantık:
    - frame < start_frame: parçacık görünmez
    - frame >= start_frame: parçacık yukarıdan girer ve aşağı iner
    - z <= z_floor olduğunda parçacık görünmez olur ve video sonuna kadar geri gelmez
    - sahneye sürekli yeni parçacık girişi, farklı start_frame değerleriyle oluşturulan
      farklı artist'ler üzerinden sağlanır

    Hareket:
        U = r F(eta)
        V = r G(eta)
        W = delta^2 H(eta)
    """
    x = np.zeros(nframes)
    y = np.zeros(nframes)
    z_hist = np.zeros(nframes)
    r_hist = np.zeros(nframes)
    vis_hist = np.zeros(nframes)

    r = float(r0)
    th = float(th0)
    z = float(z0)

    dt_sub = dt / substeps
    alive = False

    for i in range(nframes):
        if i < start_frame:
            x[i] = r * np.cos(th)
            y[i] = r * np.sin(th)
            z_hist[i] = z
            r_hist[i] = r
            vis_hist[i] = 0.0
            continue

        if i == start_frame:
            alive = True

        if not alive:
            x[i] = r * np.cos(th)
            y[i] = r * np.sin(th)
            z_hist[i] = z
            r_hist[i] = r
            vis_hist[i] = 0.0
            continue

        # Mevcut konumu kaydet.
        x[i] = r * np.cos(th)
        y[i] = r * np.sin(th)
        z_hist[i] = z
        r_hist[i] = r

        age = i - start_frame

        fade_in = np.clip(age / fade_in_frames, 0.0, 1.0)
        fade_out = np.clip((z - z_floor) / fade_out_height, 0.0, 1.0)

        vis_hist[i] = fade_in * fade_out

        # Bir sonraki frame için konumu ilerlet.
        for _ in range(substeps):
            Fn, Gn, Hn = vk_profiles(z)

            Fn = float(Fn)
            Gn = float(Gn)
            Hn = float(Hn)

            U = r * Fn
            V_over_r = Gn
            W = DELTA2 * Hn

            dr_dt = U_VISUAL_GAIN * U
            dth_dt = THETA_VISUAL_GAIN * V_over_r
            dz_dt = W_VISUAL_GAIN * W

            r = r + dr_dt * dt_sub
            th = th + dth_dt * dt_sub
            z = z + dz_dt * dt_sub

            r = np.clip(r, r_min, r_max)

            if z <= z_floor + 0.04:
                z = z_floor
                alive = False
                break

            z = np.clip(z, z_floor, Z_max - 0.12)

    return x, y, z_hist, r_hist, vis_hist


# ============================================================
# Trajectory sampling with effective flow clock
# ============================================================
def sample_traj_at(traj, s):
    """
    Önceden hesaplanmış trajeyi kesirli efektif frame değerinde örnekler.
    Böylece spin-up sırasında parçacık hareketi de yumuşak hızlanır.
    """
    s = float(np.clip(s, 0.0, NFRAMES - 1.0))

    i0 = int(np.floor(s))
    i1 = min(i0 + 1, NFRAMES - 1)
    a = s - i0

    return tuple((1.0 - a) * arr[i0] + a * arr[i1] for arr in traj)


def trail_indices_for(s, trail_len):
    """
    Trail çizgilerinin hangi efektif frame aralığını göstereceğini verir.
    """
    idx = int(np.floor(np.clip(s, 0.0, NFRAMES - 1.0)))
    i0 = max(0, idx - trail_len)
    return i0, idx


# ============================================================
# Figure setup
# ============================================================
fig = plt.figure(figsize=(8.5, 6.3), dpi=110)
ax = fig.add_subplot(111, projection="3d")

try:
    ax.computed_zorder = False
except Exception:
    pass

fig.patch.set_facecolor("white")
ax.set_facecolor("white")


# ============================================================
# Static scene
# ============================================================
def draw_static_scene(ax):
    theta = np.linspace(0, 2 * np.pi, 280)
    r = np.linspace(0, R_disk, 115)
    Theta, Rad = np.meshgrid(theta, r)

    X = Rad * np.cos(Theta)
    Y = Rad * np.sin(Theta)

    Z_top = np.zeros_like(X)
    Z_bottom = -H_disk * np.ones_like(X)

    metal = (
        0.17
        + 0.10 * (Rad / R_disk) ** 0.90
        + 0.010 * np.cos(16 * np.pi * Rad / R_disk)
    )
    metal = np.clip(metal, 0.0, 1.0)

    face_colors_top = cm.Greys(metal)
    face_colors_top[..., 3] = DISK_TOP_ALPHA

    ax.plot_surface(
        X, Y, Z_top,
        facecolors=face_colors_top,
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=DISK_TOP_ALPHA,
        zorder=1,
    )

    ax.plot_surface(
        X, Y, Z_bottom,
        color="#5f5f5f",
        linewidth=0,
        antialiased=True,
        shade=True,
        alpha=1.0,
        zorder=1,
    )

    Theta_side, Z_side = np.meshgrid(theta, np.linspace(-H_disk, 0.0, 36))
    X_side = R_disk * np.cos(Theta_side)
    Y_side = R_disk * np.sin(Theta_side)

    ax.plot_surface(
        X_side, Y_side, Z_side,
        color="#4f4f4f",
        linewidth=0,
        antialiased=True,
        shade=True,
        alpha=1.0,
        zorder=1,
    )

    ax.plot(
        R_disk * np.cos(theta),
        R_disk * np.sin(theta),
        np.zeros_like(theta),
        color="#2f2f2f",
        lw=1.7,
        zorder=5,
    )

    ax.plot(
        R_disk * np.cos(theta),
        R_disk * np.sin(theta),
        -H_disk * np.ones_like(theta),
        color="#444444",
        lw=1.1,
        zorder=5,
    )

    for rr in np.linspace(0.8, R_disk - 0.35, 8):
        ax.plot(
            rr * np.cos(theta),
            rr * np.sin(theta),
            0.025 * np.ones_like(theta),
            color="white",
            lw=0.50,
            alpha=0.10,
            zorder=3,
        )

    # Su silindiri
    theta_c = np.linspace(0, 2 * np.pi, 190)
    z_c = np.linspace(Z_min, Z_max, 65)

    Theta_c, Z_c = np.meshgrid(theta_c, z_c)
    X_c = R_fluid * np.cos(Theta_c)
    Y_c = R_fluid * np.sin(Theta_c)

    ax.plot_surface(
        X_c, Y_c, Z_c,
        color="#6ec6ff",
        alpha=0.075,
        linewidth=0,
        antialiased=True,
        shade=False,
        zorder=0,
    )

    r_top = np.linspace(0, R_fluid, 60)
    theta_top = np.linspace(0, 2 * np.pi, 160)
    Theta_top, Rad_top = np.meshgrid(theta_top, r_top)

    X_top = Rad_top * np.cos(Theta_top)
    Y_top = Rad_top * np.sin(Theta_top)
    Z_top_water = Z_max * np.ones_like(X_top)

    ax.plot_surface(
        X_top, Y_top, Z_top_water,
        color="#aee6ff",
        alpha=0.075,
        linewidth=0,
        antialiased=True,
        shade=False,
        zorder=0,
    )

    ax.plot(
        R_fluid * np.cos(theta_c),
        R_fluid * np.sin(theta_c),
        Z_min * np.ones_like(theta_c),
        color="#2d9cdb",
        lw=1.45,
        alpha=0.78,
        zorder=4,
    )

    ax.plot(
        R_fluid * np.cos(theta_c),
        R_fluid * np.sin(theta_c),
        Z_max * np.ones_like(theta_c),
        color="#2d9cdb",
        lw=1.35,
        alpha=0.65,
        zorder=4,
    )

    for th in np.linspace(0, 2 * np.pi, 10, endpoint=False):
        ax.plot(
            [R_fluid * np.cos(th), R_fluid * np.cos(th)],
            [R_fluid * np.sin(th), R_fluid * np.sin(th)],
            [Z_min, Z_max],
            color="#56b4ef",
            lw=0.55,
            alpha=0.16,
            zorder=0,
        )

    add_transparent_box(
        ax,
        xmin=-box_half,
        xmax=box_half,
        ymin=-box_half,
        ymax=box_half,
        zmin=box_zmin,
        zmax=box_zmax,
        face_alpha=0.026,
        edge_alpha=0.22,
    )

    origin = np.array([0.0, 0.0, 0.0])

    add_arrow3d(
        ax,
        start=origin,
        end=(0.0, 0.0, 22.0),
        color="black",
        lw=2.3,
        mutation_scale=18,
    )
    ax.text(0.25, 0.20, 22.7, r"$z$", fontsize=15, color="black")

    add_arrow3d(
        ax,
        start=origin,
        end=(7.3, 0.0, 0.0),
        color="black",
        lw=2.3,
        mutation_scale=18,
    )
    ax.text(7.75, 0.0, 0.25, r"$r$", fontsize=15, color="black")

    # Statik dönme oku
    rot_R = 1.15
    rot_z = 18.5

    phi = np.linspace(np.deg2rad(35), np.deg2rad(330), 150)
    x_rot = rot_R * np.cos(phi)
    y_rot = rot_R * np.sin(phi)
    z_rot = rot_z * np.ones_like(phi)

    ax.plot(x_rot, y_rot, z_rot, color="black", lw=1.9)

    add_arrow3d(
        ax,
        start=(x_rot[-8], y_rot[-8], z_rot[-8]),
        end=(x_rot[-1], y_rot[-1], z_rot[-1]),
        color="black",
        lw=1.9,
        mutation_scale=16,
    )

    ax.set_xlim(-7.4, 8.2)
    ax.set_ylim(-7.2, 7.2)
    ax.set_zlim(-1.2, 23.5)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

    ax.grid(False)

    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis.pane.fill = False
        axis.pane.set_edgecolor((1, 1, 1, 0))
        axis.line.set_color((1, 1, 1, 0))

    ax.set_box_aspect((1.35, 1.25, 1.55))
    ax.view_init(elev=23, azim=-50)


draw_static_scene(ax)


# ============================================================
# Dynamic artists
# ============================================================

# ------------------------------------------------------------
# 1) Disk motion markers
# ------------------------------------------------------------
disk_marker_lines = []
disk_marker_offsets = np.linspace(0, 2 * np.pi, 5, endpoint=False)

for _ in range(5):
    line, = ax.plot(
        [], [], [],
        color="#111111",
        lw=1.55,
        alpha=0.62,
        solid_capstyle="round",
        zorder=900,
    )
    disk_marker_lines.append(line)

disk_rim_marker_lines = []
rim_marker_offsets = np.linspace(0, 2 * np.pi, 10, endpoint=False)

for _ in range(10):
    line, = ax.plot(
        [], [], [],
        color="#101010",
        lw=1.35,
        alpha=0.72,
        solid_capstyle="round",
        zorder=902,
    )
    disk_rim_marker_lines.append(line)

disk_marker_dot, = ax.plot(
    [], [], [],
    linestyle="None",
    marker="o",
    markersize=3.3,
    color="#101010",
    alpha=0.90,
    zorder=903,
)


# ------------------------------------------------------------
# 2) Uzak alan parçacıkları: zamanlanmış sürekli enjeksiyon
# ------------------------------------------------------------
rng = np.random.default_rng(42)

far_particle_specs = []
far_particle_artists = []
far_particle_trajs = []

far_columns = [
    (1.15, np.deg2rad(18)),
    (1.75, np.deg2rad(72)),
    (2.35, np.deg2rad(128)),
    (2.95, np.deg2rad(188)),
    (3.55, np.deg2rad(242)),
    (4.15, np.deg2rad(304)),
    (4.75, np.deg2rad(36)),
    (5.05, np.deg2rad(160)),
]

# 2A) Video başında zaten alanda bulunan parçacıklar
for ic, (r_col, th_col) in enumerate(far_columns):
    for iz, z0 in enumerate(FAR_INITIAL_Z_LEVELS):
        r0 = r_col * (0.985 + 0.030 * rng.random())
        th0 = th_col + 0.025 * rng.standard_normal()
        start_frame = 0

        far_particle_specs.append((r0, th0, z0, start_frame))

# 2B) Video boyunca yukarıdan yeni giren parçacıklar
spawn_frames = np.arange(
    FAR_INFLOW_INTERVAL_FRAMES,
    NFRAMES,
    FAR_INFLOW_INTERVAL_FRAMES,
    dtype=int
)

for start_frame in spawn_frames:
    for ic, (r_col, th_col) in enumerate(far_columns):
        # Her enjeksiyonda tüm kolonlardan parçacık vermek fazla kalabalık olabilir.
        # Kolonlar dönüşümlü aktif.
        if (ic + start_frame // FAR_INFLOW_INTERVAL_FRAMES) % 2 != 0:
            continue

        r0 = r_col * (0.985 + 0.030 * rng.random())
        th0 = th_col + 0.025 * rng.standard_normal()
        z0 = rng.uniform(FAR_INFLOW_Z_RANGE[0], FAR_INFLOW_Z_RANGE[1])

        far_particle_specs.append((r0, th0, z0, start_frame))

# Artist ve trajeler
for r0, th0, z0, start_frame in far_particle_specs:
    artist, = ax.plot(
        [], [], [],
        linestyle="None",
        marker="o",
        markersize=3.1,
        color="#0d6efd",
        alpha=0.0,
        zorder=650,
    )
    far_particle_artists.append(artist)

    traj = integrate_scheduled_far_particle_trajectory(
        r0=r0,
        th0=th0,
        z0=z0,
        start_frame=start_frame,
        nframes=NFRAMES,
        dt=DT,
        z_floor=0.38,
        r_min=0.18,
        r_max=R_fluid - 0.12,
        substeps=5,
        fade_in_frames=FAR_FADE_IN_FRAMES,
        fade_out_height=FAR_FADE_OUT_HEIGHT,
    )

    far_particle_trajs.append(traj)


# ------------------------------------------------------------
# 3) Sınır tabaka / disk üstü parçacıkları
# ------------------------------------------------------------
bl_particle_specs = []
bl_particle_artists = []
bl_particle_trajs = []

bl_z_levels = np.array([0.08, 0.15, 0.25, 0.40, 0.60, 0.85, 1.15])
bl_radii = np.array([0.75, 1.15, 1.60, 2.10, 2.70, 3.30, 3.95, 4.55, 5.00])

for iz, z0 in enumerate(bl_z_levels):
    for ir, r0 in enumerate(bl_radii):
        if r0 > 4.3:
            n_theta = 2
        else:
            n_theta = 2 if r0 < 2.2 else 3

        for k in range(n_theta):
            th0 = (
                2.0 * np.pi * k / n_theta
                + 0.54 * ir
                + 0.28 * iz
                + 0.10 * rng.random()
            )

            bl_particle_specs.append((r0, th0, z0))

            _, G0, _ = vk_profiles(z0)
            marker_size = 4.1 + 1.9 * float(G0)

            particle, = ax.plot(
                [], [], [],
                linestyle="None",
                marker="o",
                markersize=marker_size,
                markerfacecolor="#00b7ff",
                markeredgecolor="#005f88",
                markeredgewidth=0.35,
                alpha=1.0,
                zorder=1500,
            )
            bl_particle_artists.append(particle)

for r0, th0, z0 in bl_particle_specs:
    traj = integrate_particle_trajectory(
        r0=r0,
        th0=th0,
        z0=BL_PARTICLE_Z_LIFT + z0,
        nframes=NFRAMES,
        dt=DT,
        z_floor=BL_PARTICLE_Z_LIFT + 0.05,
        r_min=BL_R_INNER,
        r_max=BL_R_OUTER,
        substeps=5,
    )
    bl_particle_trajs.append(traj)


# ------------------------------------------------------------
# 4A) Uzak alan aşağı yönlü trail çizgileri
# ------------------------------------------------------------
far_trail_seed_specs = [
    (2.10, np.deg2rad(35),  18.6),
    (2.70, np.deg2rad(90),  16.8),
    (3.30, np.deg2rad(145), 15.0),
    (3.85, np.deg2rad(205), 13.2),
    (4.35, np.deg2rad(265), 11.4),
    (4.85, np.deg2rad(325), 9.6),
]

far_trail_lines = []
far_trail_trajs = []

for _ in far_trail_seed_specs:
    line, = ax.plot(
        [], [], [],
        color="#3aa8f5",
        lw=1.15,
        alpha=0.56,
        zorder=470,
    )
    far_trail_lines.append(line)

for r0, th0, z0 in far_trail_seed_specs:
    traj = integrate_particle_trajectory(
        r0=r0,
        th0=th0,
        z0=z0,
        nframes=NFRAMES,
        dt=DT,
        z_floor=0.38,
        r_min=0.18,
        r_max=R_fluid - 0.12,
        substeps=5,
    )
    far_trail_trajs.append(traj)


# ------------------------------------------------------------
# 4A-2) Vortex / spiral pathline çizgileri
# ------------------------------------------------------------
vortex_trail_seed_specs = [
    (1.35, np.deg2rad(20),  4.20),
    (2.05, np.deg2rad(85),  3.70),
    (2.85, np.deg2rad(150), 3.15),
    (3.65, np.deg2rad(215), 2.55),
    (4.35, np.deg2rad(285), 2.05),
]

vortex_trail_lines = []
vortex_trail_trajs = []

for _ in vortex_trail_seed_specs:
    line, = ax.plot(
        [], [], [],
        color="#57d7ff",
        lw=1.35,
        alpha=0.58,
        zorder=900,
    )
    vortex_trail_lines.append(line)

for r0, th0, z0 in vortex_trail_seed_specs:
    traj = integrate_particle_trajectory(
        r0=r0,
        th0=th0,
        z0=z0,
        nframes=NFRAMES,
        dt=DT,
        z_floor=0.38,
        r_min=0.18,
        r_max=R_fluid - 0.12,
        substeps=6,
    )
    vortex_trail_trajs.append(traj)


# ------------------------------------------------------------
# 4B) Azaltılmış sınır tabaka trail çizgileri
# ------------------------------------------------------------
bl_trail_seed_specs = [
    (1.05, np.deg2rad(25),  BL_PARTICLE_Z_LIFT + 1.05),
    (1.55, np.deg2rad(95),  BL_PARTICLE_Z_LIFT + 0.82),
    (2.05, np.deg2rad(165), BL_PARTICLE_Z_LIFT + 0.60),
    (3.15, np.deg2rad(240), BL_PARTICLE_Z_LIFT + 0.42),
    (3.85, np.deg2rad(315), BL_PARTICLE_Z_LIFT + 0.26),
]

bl_trail_lines = []
bl_trail_trajs = []

for _ in bl_trail_seed_specs:
    line, = ax.plot(
        [], [], [],
        color="#00aeea",
        lw=1.15,
        alpha=0.64,
        zorder=1300,
    )
    bl_trail_lines.append(line)

for r0, th0, z0 in bl_trail_seed_specs:
    traj = integrate_particle_trajectory(
        r0=r0,
        th0=th0,
        z0=z0,
        nframes=NFRAMES,
        dt=DT,
        z_floor=BL_PARTICLE_Z_LIFT + 0.05,
        r_min=BL_R_INNER,
        r_max=BL_R_OUTER,
        substeps=5,
    )
    bl_trail_trajs.append(traj)


# ------------------------------------------------------------
# 5) Kırmızı tracer parçacık + fading trail segmentleri
# ------------------------------------------------------------
RED_PARTICLE_R0 = 2.25
RED_PARTICLE_THETA0 = np.deg2rad(58.0)
RED_PARTICLE_Z0 = 15.0
RED_PARTICLE_Z_MIN = BL_PARTICLE_Z_LIFT + 0.05

red_traj = integrate_particle_trajectory(
    r0=RED_PARTICLE_R0,
    th0=RED_PARTICLE_THETA0,
    z0=RED_PARTICLE_Z0,
    nframes=NFRAMES,
    dt=DT,
    z_floor=RED_PARTICLE_Z_MIN,
    r_min=0.18,
    r_max=R_fluid - 0.12,
    substeps=6,
)

red_fade_lines = []

for i in range(RED_FADE_SEGMENTS):
    line, = ax.plot(
        [], [], [],
        color="red",
        lw=1.0,
        alpha=0.0,
        zorder=2500 + i,
    )
    red_fade_lines.append(line)

red_particle_artist, = ax.plot(
    [], [], [],
    linestyle="None",
    marker="o",
    markersize=7.5,
    markerfacecolor="red",
    markeredgecolor="#600000",
    markeredgewidth=0.8,
    alpha=1.0,
    zorder=2700,
)


# ============================================================
# Animation functions
# ============================================================
def init():
    for line in disk_marker_lines:
        line.set_data([], [])
        line.set_3d_properties([])

    for line in disk_rim_marker_lines:
        line.set_data([], [])
        line.set_3d_properties([])

    disk_marker_dot.set_data([], [])
    disk_marker_dot.set_3d_properties([])

    for artist in far_particle_artists:
        artist.set_data([], [])
        artist.set_3d_properties([])
        artist.set_alpha(0.0)

    for artist in bl_particle_artists:
        artist.set_data([], [])
        artist.set_3d_properties([])

    for line in far_trail_lines:
        line.set_data([], [])
        line.set_3d_properties([])

    for line in vortex_trail_lines:
        line.set_data([], [])
        line.set_3d_properties([])

    for line in bl_trail_lines:
        line.set_data([], [])
        line.set_3d_properties([])

    for line in red_fade_lines:
        line.set_data([], [])
        line.set_3d_properties([])
        line.set_alpha(0.0)

    red_particle_artist.set_data([], [])
    red_particle_artist.set_3d_properties([])

    return (
        disk_marker_lines
        + disk_rim_marker_lines
        + [disk_marker_dot]
        + far_particle_artists
        + bl_particle_artists
        + far_trail_lines
        + vortex_trail_lines
        + bl_trail_lines
        + red_fade_lines
        + [red_particle_artist]
    )


def update(frame):
    ax.view_init(elev=23.0, azim=-50.0)

    # Gerçek video zamanı değil, efektif akış zamanı kullanılıyor.
    # İlk 3 saniyede flow_s sabit kalır.
    flow_s = FLOW_CLOCK_FRAMES[frame]
    flow_t = FLOW_CLOCK_TIME[frame]

    # Anlık normalize hız.
    omega_factor = SPIN_FACTORS[frame]

    # --------------------------------------------------------
    # Disk marker'ları
    # --------------------------------------------------------
    rr = np.linspace(0.95, 4.85, 54)
    z_marker = 0.068

    for i, (line, offset) in enumerate(zip(disk_marker_lines, disk_marker_offsets)):
        th = OMEGA * flow_t + offset

        rr_mod = rr + 0.035 * np.sin(2.2 * rr + 0.8 * i)

        x = rr_mod * np.cos(th)
        y = rr_mod * np.sin(th)
        z = np.full_like(rr_mod, z_marker)

        line.set_data(x, y)
        line.set_3d_properties(z)

        # İlk 3 saniyede marker görünür ama dönmez.
        # Hızlanma sırasında çok hafif belirginleşsin.
        line.set_alpha(0.45 + 0.17 * omega_factor)

    rim_span = np.deg2rad(8.5)
    rim_theta_local = np.linspace(-rim_span, rim_span, 12)
    rim_radius = 5.16
    rim_z = 0.088

    for line, offset in zip(disk_rim_marker_lines, rim_marker_offsets):
        th_center = OMEGA * flow_t + offset
        ths = th_center + rim_theta_local

        x = rim_radius * np.cos(ths)
        y = rim_radius * np.sin(ths)
        z = np.full_like(ths, rim_z)

        line.set_data(x, y)
        line.set_3d_properties(z)

        line.set_alpha(0.45 + 0.27 * omega_factor)

    dot_r = 4.95
    dot_th = OMEGA * flow_t + np.pi / 8

    x_d = dot_r * np.cos(dot_th)
    y_d = dot_r * np.sin(dot_th)
    z_d = 0.090

    disk_marker_dot.set_data([x_d], [y_d])
    disk_marker_dot.set_3d_properties([z_d])
    disk_marker_dot.set_alpha(0.55 + 0.35 * omega_factor)

    # --------------------------------------------------------
    # Uzak alan parçacıkları
    # --------------------------------------------------------
    for artist, traj in zip(far_particle_artists, far_particle_trajs):
        x_now, y_now, z_now, r_now, v_now = sample_traj_at(traj, flow_s)

        artist.set_data([x_now], [y_now])
        artist.set_3d_properties([z_now])

        _, G_now, _ = vk_profiles(z_now)
        base_alpha = np.clip(0.42 + 0.46 * float(G_now), 0.34, 0.88)

        # İlk 3 saniyede parçacıklar görünebilir ama hareket etmez.
        artist.set_alpha(base_alpha * v_now)

    # --------------------------------------------------------
    # Sınır tabaka parçacıkları
    # --------------------------------------------------------
    for artist, traj in zip(bl_particle_artists, bl_particle_trajs):
        x_now, y_now, z_now, r_now = sample_traj_at(traj, flow_s)

        artist.set_data([x_now], [y_now])
        artist.set_3d_properties([z_now])

        if r_now <= EDGE_FADE_START:
            alpha_now = 1.0
        else:
            alpha_now = 1.0 - (r_now - EDGE_FADE_START) / (BL_R_OUTER - EDGE_FADE_START)
            alpha_now = np.clip(alpha_now, 0.15, 1.0)

        artist.set_alpha(alpha_now)

    # --------------------------------------------------------
    # Uzak alan trail'leri
    # --------------------------------------------------------
    for line, traj in zip(far_trail_lines, far_trail_trajs):
        xh, yh, zh, rh = traj
        i0, i1 = trail_indices_for(flow_s, FAR_TRAIL_LEN)

        line.set_data(xh[i0:i1 + 1], yh[i0:i1 + 1])
        line.set_3d_properties(zh[i0:i1 + 1])

        # İlk 3 saniyede trail çok baskın olmasın.
        line.set_alpha(0.22 + 0.34 * omega_factor)

    # --------------------------------------------------------
    # Vortex / spiral pathline çizgileri
    # --------------------------------------------------------
    for line, traj in zip(vortex_trail_lines, vortex_trail_trajs):
        xh, yh, zh, rh = traj
        i0, i1 = trail_indices_for(flow_s, VORTEX_TRAIL_LEN)

        line.set_data(xh[i0:i1 + 1], yh[i0:i1 + 1])
        line.set_3d_properties(zh[i0:i1 + 1])

        # Vortex etkisi disk hızlandıkça ortaya çıksın.
        line.set_alpha(0.08 + 0.50 * omega_factor)

    # --------------------------------------------------------
    # Sınır tabaka trail'leri
    # --------------------------------------------------------
    for line, traj in zip(bl_trail_lines, bl_trail_trajs):
        xh, yh, zh, rh = traj
        i0, i1 = trail_indices_for(flow_s, BL_TRAIL_LEN)

        line.set_data(xh[i0:i1 + 1], yh[i0:i1 + 1])
        line.set_3d_properties(zh[i0:i1 + 1])

        # Sınır tabaka izleri disk hızlandıkça belirginleşsin.
        line.set_alpha(0.12 + 0.52 * omega_factor)

    # --------------------------------------------------------
    # Kırmızı tracer fading trail
    # --------------------------------------------------------
    xr, yr, zr, rr = red_traj

    for line in red_fade_lines:
        line.set_data([], [])
        line.set_3d_properties([])
        line.set_alpha(0.0)

    red_i = int(np.floor(np.clip(flow_s, 0.0, NFRAMES - 1.0)))

    n_available = max(0, red_i)
    n_segments = min(RED_FADE_SEGMENTS, n_available)

    start_idx = max(0, red_i - n_segments)

    for j in range(n_segments):
        idx0 = start_idx + j
        idx1 = idx0 + 1

        if idx1 > red_i:
            break

        line = red_fade_lines[j]

        line.set_data(xr[idx0:idx1 + 1], yr[idx0:idx1 + 1])
        line.set_3d_properties(zr[idx0:idx1 + 1])

        q = (j + 1) / max(1, n_segments)

        line.set_alpha((0.10 + 0.82 * q) * omega_factor)
        line.set_linewidth(0.85 + 1.85 * q)

    x_red, y_red, z_red, r_red = sample_traj_at(red_traj, flow_s)

    red_particle_artist.set_data([x_red], [y_red])
    red_particle_artist.set_3d_properties([z_red])

    # Kırmızı parçacık ilk 3 saniyede görünür ama izi oluşmaz.
    red_particle_artist.set_alpha(0.75 + 0.25 * omega_factor)

    return (
        disk_marker_lines
        + disk_rim_marker_lines
        + [disk_marker_dot]
        + far_particle_artists
        + bl_particle_artists
        + far_trail_lines
        + vortex_trail_lines
        + bl_trail_lines
        + red_fade_lines
        + [red_particle_artist]
    )


ani = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    init_func=init,
    interval=1000 / FPS,
    blit=False,
    repeat=True,
)


# ============================================================
# Save and display GIF + MP4
# ============================================================
ani.save(
    OUTPUT_GIF,
    writer=PillowWriter(fps=FPS),
    dpi=85,
)

try:
    mp4_writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=3000,
        extra_args=["-pix_fmt", "yuv420p"]
    )

    ani.save(
        OUTPUT_MP4,
        writer=mp4_writer,
        dpi=120,
    )

    mp4_ok = True

except Exception as e:
    mp4_ok = False
    print("[UYARI] MP4 kaydedilemedi.")
    print("Hata:", e)
    print("GIF yine kaydedildi. MP4 için imageio-ffmpeg/ffmpeg ayarlarını kontrol et.")

plt.close(fig)

print(f"GIF kaydedildi: {OUTPUT_GIF}")

if mp4_ok:
    print(f"MP4 kaydedildi: {OUTPUT_MP4}")

display(Image(filename=OUTPUT_GIF))

if mp4_ok:
    display(Video(OUTPUT_MP4, embed=True))